# Conditional Edge

## 조건부 Edge(`Conditional Edge`) 사용해 분기 만들기
* `Conditional Edge`를 이용해 상황에 따라 다른 노드로 연결되도록 구현할 수 있습니다.

In [10]:
from typing import Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph.message import add_messages

from langchain_core.messages import BaseMessage, HumanMessage

import time

class AgentState(BaseModel):
    # Annotated[타입, 메타데이터] -> 해당 변수에 '메타데이터'에 해당하는 특성(기능)을 부여함
    # add_messages -> message에 적용하는 메타데이터, Node에서 history에 추가할 항목만 반환해도 알아서 리스트에 추가해 줌
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list) # 맨 처음 생성 시 빈 리스트로 시작
    last_player: Literal['A', 'B']
    number: int
    end: bool

In [19]:
def node_A(state: AgentState) -> dict:
    num = state.number
    new_num = num * 2
    time.sleep(1)
    return {
        'chat_history': [HumanMessage(name='A', content=f"숫자를 {num}에서 {new_num}(으)로 바꿈")],
        'number': new_num,
        'last_player': 'A'
    }

In [21]:
def node_B(state: AgentState) -> dict:
    num = state.number
    new_num = num - 1
    time.sleep(1)

    return {
        'chat_history': [HumanMessage(name='B', content=f"숫자를 {num}에서 {new_num}(으)로 변경")],
        'number': new_num,
        'last_player': 'B'
    }

In [22]:
def node_judge(state: AgentState) -> dict:
    end = True if state.number > 100 else False
    return {
        'chat_history': [HumanMessage(name='judge', content=f"최종 숫자 {state.number}(으)로 종료합니다" if end else "계속 진행하세요")],
        'end': end
    }

In [23]:
from langgraph.graph import START, END, StateGraph

workflow = StateGraph(AgentState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('judge', node_judge)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'judge')
workflow.add_edge('B', 'judge')

## `route` 만들고 `conditional edge`에 부여하기
* `route`는 조건에 따라서 달라지는 목적지 `node`명 을 반환하는 함수입니다
* `workflow.add_conditional_edges`함수에 `route`를 적용하면, route가 반환한 `node`명으로 라우팅됩니다.

In [24]:
def judge_route(state: AgentState):
    # end가 True면 END로,
    if state.end:
        return END
    elif state.last_player == 'B':
        return 'A'
    else:
        return 'B'

workflow.add_conditional_edges(
    'judge', 
    judge_route
)

In [25]:
app = workflow.compile()

init_input = AgentState(chat_history=[], last_player='A', number=2, end=False)

for chunk in app.stream(init_input):
    print(chunk)

{'A': {'chat_history': [HumanMessage(content='숫자를 2에서 4(으)로 바꿈', additional_kwargs={}, response_metadata={}, name='A', id='2d86a848-d3b8-45fa-a226-2aba81e23a67')], 'number': 4, 'last_player': 'A'}}
{'judge': {'chat_history': [HumanMessage(content='계속 진행하세요', additional_kwargs={}, response_metadata={}, name='judge', id='08334f4b-48eb-46aa-8fd3-2f478c6924ad')], 'end': False}}
{'B': {'chat_history': [HumanMessage(content='숫자를 4에서 3(으)로 변경', additional_kwargs={}, response_metadata={}, name='B', id='632b5f20-5880-41be-be44-ed2c018ddaad')], 'number': 3, 'last_player': 'B'}}
{'judge': {'chat_history': [HumanMessage(content='계속 진행하세요', additional_kwargs={}, response_metadata={}, name='judge', id='01fa937c-7b2e-45de-970d-7230b043b9ee')], 'end': False}}
{'A': {'chat_history': [HumanMessage(content='숫자를 3에서 6(으)로 바꿈', additional_kwargs={}, response_metadata={}, name='A', id='faeb41f6-354e-47f9-bfb0-ed7c20e29880')], 'number': 6, 'last_player': 'A'}}
{'judge': {'chat_history': [HumanMessage(content

## 토마토 게임 만들기
* ai와 토마토 게임을 진행할 수 있는 LangGraph 기반 게임을 만들어 봅시다
* 규칙: 두 명이 번갈아가면서 '토마토'를 한글자씩 말합니다. 올바르게 말하지 못하면 패배합니다.

In [39]:
TOMATO = ('토', '마', '토')  # 토마토를 한 글자씩 반복

def next_expected(answer_state: str) -> str:
    """지금까지 맞게 말한 글자 수를 기준으로 다음에 나와야 할 글자"""
    return TOMATO[len(answer_state) % len(TOMATO)]

In [40]:
from pydantic import BaseModel, Field
from typing import Literal

class TomatoGameState(BaseModel):
    next_player: Literal['user', 'ai']
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    player_input: str = ""
    answer_state: str = ""
    correct: bool = True
    game_over: bool = False

In [41]:
def user_node(state: TomatoGameState) -> dict:
    user_input = input("당신의 대답: ").strip()
    print(f"사용자: {user_input}")
    return {'player_input': user_input}

def ai_node(state: TomatoGameState) -> dict:
    ai_input = next_expected(state.answer_state)
    print(f"AI: {ai_input}")
    return {'player_input': ai_input}

def judge_node(state: TomatoGameState) -> dict:
    # 토마:토  마토:토  토토:마  (len % 3 기준으로 다음 글자 판정)
    expected = next_expected(state.answer_state)
    player_input = state.player_input.strip()
    is_correct = player_input == expected

    if is_correct:
        new_answer = state.answer_state + player_input
        next_player = 'ai' if state.next_player == 'user' else 'user'
        return {
            'chat_history': [HumanMessage(
                name='judge',
                content=f"정답! 누적: {new_answer} (다음 글자: {next_expected(new_answer)})"
            )],
            'answer_state': new_answer,
            'next_player': next_player,
            'correct': True,
            'game_over': False,
        }

    loser = '사용자' if state.next_player == 'user' else 'AI'
    return {
        'chat_history': [HumanMessage(
            name='judge',
            content=f"오답! {loser} 패배 (정답: {expected}, 입력: {player_input})"
        )],
        'correct': False,
        'game_over': True,
    }

In [42]:
from langgraph.graph import START, END, StateGraph

tomato_workflow = StateGraph(TomatoGameState)

tomato_workflow.add_node('user', user_node)
tomato_workflow.add_node('ai', ai_node)
tomato_workflow.add_node('judge', judge_node)

def start_route(state: TomatoGameState):
    return 'user' if state.next_player == 'user' else 'ai'

def judge_route(state: TomatoGameState):
    if state.game_over:
        return END
    return 'user' if state.next_player == 'user' else 'ai'

tomato_workflow.add_conditional_edges(START, start_route)
tomato_workflow.add_edge('user', 'judge')
tomato_workflow.add_edge('ai', 'judge')
tomato_workflow.add_conditional_edges('judge', judge_route)

tomato_app = tomato_workflow.compile()

In [44]:
init_state = TomatoGameState(next_player='user')

for chunk in tomato_app.stream(init_state):
    print(chunk)

사용자: 토
{'user': {'player_input': '토'}}
{'judge': {'chat_history': [HumanMessage(content='정답! 누적: 토 (다음 글자: 마)', additional_kwargs={}, response_metadata={}, name='judge', id='b497583f-cb9f-4ccc-9b9f-77b84f89d1d9')], 'answer_state': '토', 'next_player': 'ai', 'correct': True, 'game_over': False}}
AI: 마
{'ai': {'player_input': '마'}}
{'judge': {'chat_history': [HumanMessage(content='정답! 누적: 토마 (다음 글자: 토)', additional_kwargs={}, response_metadata={}, name='judge', id='095e729b-c363-48f4-990c-b63b75e11c1b')], 'answer_state': '토마', 'next_player': 'user', 'correct': True, 'game_over': False}}
사용자: 토
{'user': {'player_input': '토'}}
{'judge': {'chat_history': [HumanMessage(content='정답! 누적: 토마토 (다음 글자: 토)', additional_kwargs={}, response_metadata={}, name='judge', id='895a56c8-9f0f-4d20-9922-cf06b183329c')], 'answer_state': '토마토', 'next_player': 'ai', 'correct': True, 'game_over': False}}
AI: 토
{'ai': {'player_input': '토'}}
{'judge': {'chat_history': [HumanMessage(content='정답! 누적: 토마토토 (다음 글자: 마)'

### 팬아웃(Fan-Out) 구현하기
* 하나의 노드에서 여러 노드로 갈라지고(팬아웃) 다시 한 노드로 흐름을 모을 (팬인) 수 있습니다

In [45]:
from pydantic import BaseModel
from typing import Optional, Any

class FanOutState(BaseModel):
    value1: Optional[Any] = None
    value2: Optional[Any] = None

In [46]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"B 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value2 + 1}

In [29]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')

app = workflow.compile()

In [30]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)
B 노드 실행 중... (3/3)


{'value1': 1, 'value2': 1}

## 팬인(Fan-In) 구현하기
* `add_edge` 함수의 출발점은 여러 노드의 배열도 입력받을 수 있습니다.

In [33]:
def node_D(state: FanOutState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value1 + 1}

In [34]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [35]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)B 노드 실행 중... (3/3)

D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 1, 'value2': 2}

### Q. 팬인 시 노드 종료 시점이 다르다면?

In [47]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict: # 두 노드의 실행 시간을 다르게 하고 결과를 확인해보세요.
    for i in range(1, 4):
        print(f"B 노드 실행 중... ({i}/3)")
        time.sleep(3)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value2 + 1}

# def node_A(state: FanOutState) -> dict:
#     print("A 노드에서 출발합니다.")
#     return {"from_node": "A"}

# def node_B(state: FanOutState) -> dict:
#     for i in range(1, 4):
#         print(f"B 노드 실행 중... ({i}/3)")
#         time.sleep(1)
#     return {"value1": state.value1 + 1}

# def node_C(state: FanOutState) -> dict:
#     for i in range(1, 4):
#         print(f"C 노드 실행 중... ({i}/3)")
#         time.sleep(1)
#     return {"value2": state.value2 + 1}

In [48]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [49]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
C 노드 실행 중... (2/3)
C 노드 실행 중... (3/3)
B 노드 실행 중... (2/3)
B 노드 실행 중... (3/3)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 1, 'value2': 2}

# `Reducer` 활용하기

In [52]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 6):
        print(f"B 노드 실행 중... ({i}/5)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

    

In [53]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [54]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)
C 노드 실행 중... (2/3)
B 노드 실행 중... (3/5)
C 노드 실행 중... (3/3)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)


InvalidUpdateError: At key 'value1': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE

In [55]:
from typing import Annotated

def numberAddReducer(left: int, right: int) -> int: # left: 기존 State, right: return 받은 state
    return left + right # return 받은 값으로 대체하지 않고, 기존 값에 return 받은 값을 더함

class ReducerState(BaseModel):
    value1: Annotated[Optional[Any], numberAddReducer] = None
    value2: Optional[Any] = None

In [56]:
def node_B(state: ReducerState) -> dict: # state 정의도 모두 변경
    for i in range(1, 6):
        print(f"B 노드 실행 중... ({i}/5)")
        time.sleep(1)
    return {"value1": 1} # 새로 더해줄 값만 반환하면 됨

def node_C(state: ReducerState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": 1}

def node_D(state: ReducerState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": 1}


In [57]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [58]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)
B 노드 실행 중... (3/5)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 2, 'value2': 1}

### 동적 팬아웃 구현하기
* `Conditional Edge`를 이용하면 Fan-Out을 동적으로 설정할 수 있습니다
* Fan-Out을 설정하려면, 목적지 노드 이름을 리스트로 모두 전달하면 됩니다.

In [59]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')

def fanout_router(state: ReducerState):
    if state.value2 > 0:
        return 'B'
    else:
        return ['B', 'C']
    
workflow.add_conditional_edges('A', fanout_router)

workflow.add_edge(['B', 'C'], 'D')

def end_router(state: ReducerState):
    if state.value1 > 2:
        return END
    else:
        return 'A'

workflow.add_conditional_edges('D', end_router)


app = workflow.compile()

In [60]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)C 노드 실행 중... (2/3)

B 노드 실행 중... (3/5)C 노드 실행 중... (3/3)

B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)
A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
B 노드 실행 중... (2/5)
B 노드 실행 중... (3/5)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)


{'value1': 3, 'value2': 1}

### 라우팅 심화 문제
* Fan-Out과 Reducing, 조건부 Edge를 적극적으로 활용해 음식점 운영을 최적화해봅시다 (괄호 안의 숫자는 소요시간(초))
```
burger: patty(5), bun(3) 각각 필요 -> burgerSet(2)
fries: fry(3) -> friesSet(1)
beverage: beverage(2)

세 가지가 모두 준비되면 종료
```

In [61]:
from pydantic import BaseModel
from typing import List, Annotated, Literal, Optional

def menuReducer(left:List[str], right:List[str]) -> List[str]:
    return left + right

class RestaurantState(BaseModel):
    menu_ordered: List[Literal['burger', 'fries', 'beverage']]
    menu_complete: Annotated[Optional[List[Literal['burger', 'fries', 'beverage']]], menuReducer] = []


In [62]:
import time

# 각종 그래프 노드 실행 및 상태 갱신 통합: 반환값으로 state의 menu_complete를 적절히 업데이트하도록 구성
def g_burger_start(state: RestaurantState) -> dict:
    print("버거 준비 시작")
    return {}

def g_fries_start(state: RestaurantState) -> dict:
    print("감자튀김 준비 시작")
    return {}

def g_patty(state: RestaurantState) -> dict:
    print("패티 조리 시작")
    time.sleep(5)
    print("패티 조리 완료")
    return {}

def g_bun(state: RestaurantState) -> dict:
    print("번 조리 시작")
    time.sleep(3)
    print("번 조리 완료")
    return {}

def g_burger_set(state: RestaurantState) -> dict:
    print("버거 세트 조립 시작")
    time.sleep(2)
    print("버거 세트 조립 완료")
    return {}

def g_burger_complete(state: RestaurantState) -> dict:
    print("버거 완성")
    return {'menu_complete': ['burger']}

def g_fry(state: RestaurantState) -> dict:
    print("감자튀김 조리 시작")
    time.sleep(3)
    print("감자튀김 조리 완료")
    return {}

def g_fries_set(state: RestaurantState) -> dict:
    print("감자튀김 세트 준비 시작")
    time.sleep(1)
    print("감자튀김 세트 준비 완료")
    return {}

def g_fries_complete(state: RestaurantState) -> dict:
    print("감자튀김 완성")
    return {'menu_complete': ['fries']}

def g_beverage(state: RestaurantState) -> dict:
    print("음료 준비 시작")
    time.sleep(2)
    print("음료 준비 완료")
    return {'menu_complete': ['beverage']}

#### 2단계: 파이프라인 내부 고정 Edge 연결
* 버거: `burger_start` → `patty`/`bun` (Fan-Out) → `burger_set` (Fan-In) → `burger_complete`
* 감튀: `fries_start` → `fry` → `fries_set` → `fries_complete`
* 음료: `beverage` 단독 노드 (START 연결은 3단계에서)

In [63]:
from langgraph.graph import StateGraph, START, END

restaurant_workflow = StateGraph(RestaurantState)

# 노드 등록
restaurant_workflow.add_node('burger_start', g_burger_start)
restaurant_workflow.add_node('patty', g_patty)
restaurant_workflow.add_node('bun', g_bun)
restaurant_workflow.add_node('burger_set', g_burger_set)
restaurant_workflow.add_node('burger_complete', g_burger_complete)

restaurant_workflow.add_node('fries_start', g_fries_start)
restaurant_workflow.add_node('fry', g_fry)
restaurant_workflow.add_node('fries_set', g_fries_set)
restaurant_workflow.add_node('fries_complete', g_fries_complete)

restaurant_workflow.add_node('beverage', g_beverage)

# [버거 파이프라인] Fan-Out → Fan-In
restaurant_workflow.add_edge('burger_start', 'patty')
restaurant_workflow.add_edge('burger_start', 'bun')
restaurant_workflow.add_edge(['patty', 'bun'], 'burger_set')
restaurant_workflow.add_edge('burger_set', 'burger_complete')

# [감자튀김 파이프라인]
restaurant_workflow.add_edge('fries_start', 'fry')
restaurant_workflow.add_edge('fry', 'fries_set')
restaurant_workflow.add_edge('fries_set', 'fries_complete')

# [음료 파이프라인] beverage는 단일 노드 (3단계 START 라우터에서 연결)

#### 3단계: START conditional router (동적 Fan-Out)
* `menu_ordered`에 따라 주문된 파이프라인만 병렬로 시작
* 반환값이 리스트이면 여러 노드로 동시에 Fan-Out

In [64]:
def start_router(state: RestaurantState):
    routes = []
    if 'burger' in state.menu_ordered:
        routes.append('burger_start')
    if 'fries' in state.menu_ordered:
        routes.append('fries_start')
    if 'beverage' in state.menu_ordered:
        routes.append('beverage')

    if len(routes) == 1:
        return routes[0]
    return routes

restaurant_workflow.add_conditional_edges(START, start_router)

#### 4단계: 각 파이프라인 끝 → END 연결
* 주문된 파이프라인이 각각 끝나면 해당 브랜치 종료
* 모든 브랜치가 END에 도달하면 그래프 전체 종료

In [65]:
restaurant_workflow.add_edge('burger_complete', END)
restaurant_workflow.add_edge('fries_complete', END)
restaurant_workflow.add_edge('beverage', END)

#### 5단계: compile & 실행
* 세 메뉴를 모두 주문하면 세 파이프라인이 병렬로 실행됩니다

In [ ]:
restaurant_app = restaurant_workflow.compile()

init_state = RestaurantState(
    menu_ordered=['burger', 'fries', 'beverage']
    # menu_ordered=['burger', 'fries']
)
result = restaurant_app.invoke(init_state)
result

버거 준비 시작
감자튀김 준비 시작
번 조리 시작
감자튀김 조리 시작
패티 조리 시작
번 조리 완료
감자튀김 조리 완료
패티 조리 완료
버거 세트 조립 시작
감자튀김 세트 준비 시작
감자튀김 세트 준비 완료
버거 세트 조립 완료
버거 완성
감자튀김 완성


{'menu_ordered': ['burger', 'fries'], 'menu_complete': ['burger', 'fries']}

## 음식점 Workflow 전체 구조

![음식점 Workflow 전체 구조](image/restaurant_workflow.png)

```mermaid
flowchart TD
    START([START]) --> ROUTER{start_router<br/>menu_ordered 확인}

    ROUTER -->|burger 주문| BS[burger_start]
    ROUTER -->|fries 주문| FS[fries_start]
    ROUTER -->|beverage 주문| BV[beverage<br/>2초]

    subgraph BURGER["버거 파이프라인"]
        BS --> P[patty<br/>5초]
        BS --> BN[bun<br/>3초]
        P --> BS_SET[burger_set<br/>2초]
        BN --> BS_SET
        BS_SET --> BC[burger_complete]
    end

    subgraph FRIES["감자튀김 파이프라인"]
        FS --> FR[fry<br/>3초]
        FR --> FSET[fries_set<br/>1초]
        FSET --> FC[fries_complete]
    end

    BC --> END_NODE([END])
    FC --> END_NODE
    BV --> END_NODE
```

---

### 핵심 패턴 3가지

#### 1. 동적 Fan-Out (START)
```
START ──→ start_router ──┬──→ burger_start
                         ├──→ fries_start
                         └──→ beverage
```
`menu_ordered`에 있는 메뉴만 골라 **동시에** 시작합니다.

#### 2. Fan-Out → Fan-In (버거)
```
burger_start ──┬──→ patty (5초) ──┐
               └──→ bun   (3초) ──┴──→ burger_set (2초) → burger_complete
```
`patty`와 `bun`이 **병렬** 실행되고, **둘 다 끝나야** `burger_set`이 시작됩니다.

#### 3. 병렬 브랜치 → END
```
burger_complete ──┐
fries_complete  ──┼──→ END
beverage        ──┘
```
각 파이프라인이 독립적으로 끝나고, **모든 브랜치가 END에 도달**하면 그래프 전체가 종료됩니다.

---

### 시간 흐름 (3메뉴 전체 주문 시)

```
시간(초)  0    2    3    5    7   10
          │    │    │    │    │    │
beverage  ████                          (2초)
fries          ████████                  (3+1=4초)
burger    ████████████████████████████  (5+2=7초, bun 3초 후 patty 5초 대기)
                                        ↑
                                   전체 완료 ≈ 10초
```

세 파이프라인이 **동시에** 시작되므로, 총 소요 시간은 가장 긴 버거 파이프라인 기준 **약 10초**입니다. 순차 실행이었다면 16초가 걸렸을 것입니다.